# Code vs Code - stage 1: can a browser-sized model build it?

Each candidate model plays the sample challenge ("Team notes") through its scripted
conversations, with exactly the prompt the game will give it, and the app it ends with is run
against the 42 hidden tests.

Second run. The first scored 0 of 42 for every candidate - mostly imports and the app's shape,
not coding. Since then: the starter is a scaffold (routing, body parsing and answers done; the
helper fills in each route), the helper is told what the runtime has, and after the scripted
messages up to `REPAIR_TURNS` more turns pass back what the workspace would show a player when
the app does not run - load errors, schema errors, crashes. Never the hidden tests.

- **Part A** - the browser-sized candidates at full precision under `transformers` (the
  ceiling: a 4-bit copy can only do as well or worse), and a **teacher**-sized model in
  4-bit. A teacher writes cvchelper's training conversations, and a conversation is only kept
  if its app passes the hidden tests - so a teacher has to pass, or the plan for training data
  changes.
- **Part B** - parked. The finalists converted by us to a 4-bit browser format and measured
  again. MLC's converter crashed inside TVM; it returns as an ONNX export for transformers.js
  once a model passes Part A.

Run on a **T4 GPU** (Runtime > Change runtime type). The small models take roughly 10 to 20
minutes each, the 7B teacher around twenty. About an hour in all, which fits a free-tier
session. Set `RUN_TEACHERS = False` to skip the teacher. If a model runs out of GPU memory it is
skipped and the rest carry on; the last cell zips whatever finished. Everything is kept on the Colab machine under `/content/stage1/`,
and the last step packs it into zip files to download: the results (tables, every reply, every
app the models wrote) and, once Part B is back, the 4-bit model. Nothing is written to Google Drive.

A Colab session that disconnects loses its files, so download the zips before closing the tab.

No API key of any kind is used. Models are downloaded from public Hugging Face repos.

## Settings

In [ ]:
REPO = "https://github.com/MoAbboud/PortFolio.git"
BRANCH = "code-vs-code-updates"   # the branch with codevscode/ stage 1; "main" once merged

# Browser-sized: could ship as cvchelper's base. Loaded at 16 bits.
# SmolLM2-1.7B is out: its context is 8,192 tokens and our prompt reached 8,776 (run 2).
CANDIDATES = [
    "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "Qwen/Qwen3-1.7B",
]
# Teacher-sized: too big for a browser, there to write training data. Loaded at 4 bits
# (bitsandbytes), which is how they fit a T4's 15 GB. The 14B is out on the free tier: it
# loaded, then ran out of GPU memory in attention on our prompt (run 2).
TEACHERS = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",
]
RUN_TEACHERS = True
SEEDS = [0]            # 0 is greedy. Add 1, 2 for sampled runs, to see how much results move.
MAX_NEW_TOKENS = 4096  # a whole app.js is around 2,500 tokens
REPAIR_TURNS = 2       # extra turns passing back load errors, as a player would

## Setup: GPU, the repo, Node and the challenge

In [ ]:
# Stop here without a GPU. On a CPU every model fails to load or takes hours a turn, and the run
# ends with an empty table. On the free tier the GPU comes back once the usage limit resets.
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU. If Colab says the GPU "
                     "limit is reached, wait for the free tier to reset (usually within a day) and run again.")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.isdir("PortFolio"):
    !git clone --depth 1 --branch {BRANCH} {REPO} PortFolio
else:
    !cd PortFolio && git pull --ff-only

In [ ]:
# Node 22 or newer: the hidden tests run in the same JavaScript runner as the game.
!node --version 2>/dev/null | grep -qE '^v(2[2-9]|[3-9][0-9])' || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null && apt-get install -y nodejs >/dev/null)
!node --version
!cd PortFolio/codevscode && npm ci --silent
!cd PortFolio/codevscode && node tools/validate.mjs challenges/sample

In [ ]:
import sys, json, time, datetime, pathlib
sys.path.insert(0, "/content/PortFolio/codevscode")
from codevscode.harness import load_challenge, measure

challenge = load_challenge("/content/PortFolio/codevscode/challenges/sample")
print(challenge.id, "-", list(challenge.scripts), "-", len(challenge.answers), "hidden requirements")

RUN = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
OUT = pathlib.Path(f"/content/stage1/{RUN}")   # on the Colab machine only
OUT.mkdir(parents=True, exist_ok=True)
print("saving to", OUT)

In [ ]:
def save(rows, part):
    """The table as JSON, and for every row the replies and the files the model ended with."""
    table = []
    for row in rows:
        played = row["replay"]
        folder = OUT / part / row["model"].replace("/", "__").replace(" ", "_") / row["script"]
        folder.mkdir(parents=True, exist_ok=True)
        for name, text in played.files.items():
            (folder / name).write_text(text, encoding="utf-8")
        (folder / "conversation.json").write_text(json.dumps(
            [{"message": t.message, "reply": t.reply, "wrote": t.wrote, "skipped": t.skipped,
              "seconds": round(t.seconds, 1), "repair": t.repair} for t in played.turns], indent=2), encoding="utf-8")
        table.append({k: v for k, v in row.items() if k != "replay"})
    (OUT / f"{part}.json").write_text(json.dumps(table, indent=2), encoding="utf-8")
    return table

COLUMNS = ["model", "script", "passed", "of", "loads", "repairs", "correctness", "security", "efficiency", "practicality", "seconds"]

## Part A - the candidates, then the teachers

Candidates at 16 bits, teachers at 4 bits. Qwen3 runs with its thinking mode off: the helper
should write code, not a long chain of reasoning first.

In [ ]:
# bitsandbytes loads the teachers at 4 bits. Installed here, not at the top, so a problem with
# it can only affect the teachers.
if RUN_TEACHERS:
    !pip install -q -U bitsandbytes

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load(name, four_bit=False):
    tokenizer = AutoTokenizer.from_pretrained(name)
    if four_bit:
        quantisation = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                          bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(name, quantization_config=quantisation, device_map="cuda")
    else:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="cuda")
    model.eval()
    return tokenizer, model

def make_generate(name, tokenizer, model, seed):
    extra = {"enable_thinking": False} if "Qwen3" in name else {}
    def generate(messages):
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, **extra)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        torch.manual_seed(seed)
        sampling = {"do_sample": True, "temperature": 0.7, "top_p": 0.95} if seed else {"do_sample": False}
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **sampling)
        return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generate

def report(row):
    print(f"  {row['script']:16} {row['passed']:>2}/{row['of']}  loads={row['loads']}  repairs={row['repairs']}  "
          f"C {row['correctness']}  S {row['security']}  E {row['efficiency']}  P {row['practicality']}  "
          f"{row['seconds']} s")

RUNS = [(name, False) for name in CANDIDATES] + ([(name, True) for name in TEACHERS] if RUN_TEACHERS else [])

rows_a = []
for name, four_bit in RUNS:
    label = f"{name}{' 4-bit' if four_bit else ''}"
    print(f"\n=== {label}")
    started = time.time()
    try:
        tokenizer, model = load(name, four_bit)
    except Exception as error:   # a teacher that does not load must not lose the rows so far
        print(f"  could not load {label}: {error}")
        continue
    try:
        for seed in SEEDS:
            for row in measure(challenge, make_generate(name, tokenizer, model, seed), f"{label} seed {seed}",
                               repair_turns=REPAIR_TURNS):
                rows_a.append(row)
                report(row)
    except torch.cuda.OutOfMemoryError as error:   # skip this model, keep every row so far
        print(f"  {label} ran out of GPU memory and is skipped: {str(error).splitlines()[0]}")
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    print(f"  ({time.time() - started:.0f} s)")
    save(rows_a, "part-a")

In [ ]:
import pandas as pd
table_a = pd.DataFrame(save(rows_a, "part-a"))
if table_a.empty:
    print("No results: no model finished. Look above for 'could not load' or 'ran out of GPU memory'.")
else:
    display(table_a[COLUMNS])

## Part B - parked

Converting to MLC's 4-bit browser format crashed inside TVM (a segfault while compiling the
quantisation step, MLC nightly with TVM 0.26.dev246) on the GPU and on the CPU, under Python
3.13 and 3.12 alike. The bug is in that build, not in anything here.

Part B comes back once a model passes Part A, through the plan's other browser runtime: an
ONNX export, quantised to 4 bits and run by transformers.js, which needs no TVM at all.

## Last step - the zips to download

Two files, so the small one arrives even if the big one is slow:

- `cvc-stage1-results-<run>.zip` - the tables, every conversation, every app the models wrote.
  A few MB.
- `cvc-stage1-model-<run>.zip` - the 4-bit model folder(s) from Part B, exactly what a browser
  loads. Several hundred MB for a 1.5B model. Only made if Part B ran.

Your browser asks where to save each one. If a download stalls, the same files are in the
file panel on the left under `/content/` - right-click, Download.

In [ ]:
import shutil
from google.colab import files

results_zip = shutil.make_archive(f"/content/cvc-stage1-results-{RUN}", "zip", OUT)
print(f"{results_zip}: {pathlib.Path(results_zip).stat().st_size / 1e6:.1f} MB")

model_zip = None
if globals().get("BUILDS"):
    staging = pathlib.Path(f"/content/cvc-model-{RUN}")
    for build in BUILDS:
        shutil.copytree(build, staging / pathlib.Path(build).name, dirs_exist_ok=True)
    model_zip = shutil.make_archive(f"/content/cvc-stage1-model-{RUN}", "zip", staging)
    print(f"{model_zip}: {pathlib.Path(model_zip).stat().st_size / 1e6:.0f} MB")
else:
    print("Part B has not run, so there is no model to zip yet.")

files.download(results_zip)
if model_zip:
    files.download(model_zip)